### Data preprocessing


In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
full = pd.read_csv(BASE + "training_data.csv")
test = pd.read_csv(BASE + "testing_data.csv") # scellé jusqu'à la Partie F
full["id"], test["id"] = range(len(full)), range(len(test))


In [23]:
print(full.isnull().sum())

tweet        0
sarcasm      0
sentiment    0
dialect      0
id           0
dtype: int64


### full Dups, tweet Dups

In [24]:
full._dups = full.duplicated()
print(f"Number of duplicate lines in the full set: {full._dups.sum()}")

Number of duplicate lines in the full set: 0


C:\Users\user\AppData\Local\Temp\ipykernel_9628\175053858.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  full._dups = full.duplicated()


In [25]:
tweet_dups = full["tweet"].duplicated(keep="first")
print(f"Number of duplicate tweets in the training set: {tweet_dups.sum()}")


Number of duplicate tweets in the training set: 61


In [26]:
dup_tweets = []

def show_dup_tweets(df):
    for index, row in df.iterrows():
        if df.duplicated(subset=["tweet"], keep=False).iloc[index]:
            dup_tweets.append(row)

show_dup_tweets(full)
dup_tweets_df = pd.DataFrame(dup_tweets)
display(dup_tweets_df)

,tweet,sarcasm,sentiment,dialect,id
139,"""RT @mh_awadi: #مصر في الأخبار :أغذية ملوثةأكل...",True,NEG,egypt,139
163,"""RT @A822822A: ياروافض بلا كلام وإلا سوف أخرج♦...",True,NEG,msa,163
541,"""RT @safa_tv: #العبادي يرد على #أردوغان :بالتأ...",False,NEG,msa,541
646,"""RT @BassamJaara: حلب لن تسقط واهلها لن يغادرو...",False,NEG,msa,646
689,"""RT @SYRIA_KIDS: شكراً #بشارآثرتهم على نفسك .و...",False,NEG,msa,689
...,...,...,...,...,...
11872,"""@sawabcenter داعش مجرمين قتلة ولكن النظام الس...",True,NEG,msa,11872
11897,"""RT @HUN_0_TER: @Counselkremlin الرئيس #الروسي...",False,NEU,msa,11897
12158,"""RT @ajmubasher: وزير خارجية #بريطانيا يدعو لل...",False,NEU,msa,12158
12280,"""RT @M__albugaily: هاجموا الحبيبة مصر بضراوة ل...",False,NEG,egypt,12280


In [27]:
display(dup_tweets_df)

,tweet,sarcasm,sentiment,dialect,id
139,"""RT @mh_awadi: #مصر في الأخبار :أغذية ملوثةأكل...",True,NEG,egypt,139
163,"""RT @A822822A: ياروافض بلا كلام وإلا سوف أخرج♦...",True,NEG,msa,163
541,"""RT @safa_tv: #العبادي يرد على #أردوغان :بالتأ...",False,NEG,msa,541
646,"""RT @BassamJaara: حلب لن تسقط واهلها لن يغادرو...",False,NEG,msa,646
689,"""RT @SYRIA_KIDS: شكراً #بشارآثرتهم على نفسك .و...",False,NEG,msa,689
...,...,...,...,...,...
11872,"""@sawabcenter داعش مجرمين قتلة ولكن النظام الس...",True,NEG,msa,11872
11897,"""RT @HUN_0_TER: @Counselkremlin الرئيس #الروسي...",False,NEU,msa,11897
12158,"""RT @ajmubasher: وزير خارجية #بريطانيا يدعو لل...",False,NEU,msa,12158
12280,"""RT @M__albugaily: هاجموا الحبيبة مصر بضراوة ل...",False,NEG,egypt,12280


In [28]:
full.drop_duplicates(subset=["tweet"], keep=False, inplace=True)

In [29]:
full["tweet"].isnull().sum()

np.int64(0)

### Mapping Emojis in arabic


In [30]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ...",False,NEG,msa,5
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......,True,NEG,egypt,6
7,#انتخبوا_البرص مشكلة,True,NEG,egypt,7
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ...",False,POS,msa,8
9,▪️محافظ العاصمة تفقد سير العملية الانتخابية في...,False,NEU,msa,9


In [31]:
from arabic_emojipedia.emoji_description import get_emoji_description
import emoji

emoji = "😊"

description = get_emoji_description(emoji)

print(f"Description for {emoji}: {description}")

Description for 😊: وجه مبتسم بعيون مبتسمة


In [32]:
import emoji
from arabic_emojipedia.emoji_description import get_emoji_description

def replace_emoji_with_desc(text):

    return emoji.replace_emoji(text, replace=lambda chars,data_dict: f" {get_emoji_description(chars)} ")

full["tweet"] = full["tweet"].apply(replace_emoji_with_desc)

In [33]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب ...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ...",False,NEG,msa,5
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......,True,NEG,egypt,6
7,#انتخبوا_البرص مشكلة,True,NEG,egypt,7
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ...",False,POS,msa,8
9,مربع صغير أسود محافظ العاصمة تفقد سير العملية...,False,NEU,msa,9


### Punctuation and special characters removal

Keeping the @ and # actual values as they can be important

In [34]:
full.head

<bound method NDFrame.head of                                                    tweet  sarcasm sentiment  \
0      "د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...    False       NEU   
1      "مع فيدرر يا آجا والكبار  وجه مبتسم بعيون قلب ...    False       NEU   
2      “الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...     True       NEG   
3      "@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...     True       NEG   
4      "قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...    False       NEU   
...                                                  ...      ...       ...   
12542  #المصري_اليوم #علاء_عبدالفتاح: #مصر لم تشهد حك...    False       NEG   
12543  هلأ صاير انت يا #فلعوط بدك تعطي محاضرات ع تويت...     True       NEG   
12544  "لا اله الا الله قلب بنفسجي #أيفون_البروفيسور ...    False       NEU   
12546  RT @Yousiif65: هاري بوتر؟ نظارات  قبعة عالية  ...    False       POS   
12547        النت يلعن شكله طول الوقت تمام لين بدا الحفل     True       NEG   

      dialect     id 

In [35]:
import re

def remove_punctuation(text):
    text = re.sub(r'[^\w\s ! ?]','', text)
    text = re.sub(r'@','', text)
    text = re.sub(r'#','', text)
    text = re.sub(r'http\S+','', text)
    text = re.sub(r'www\.\S+','', text)
    text = re.sub(r'_',' ', text)
    return text

full['tweet'] = full['tweet'].apply(remove_punctuation)

full['tweet'].head(10)

0    د محمود العلايليأرى أن الفريق أحمد شفيق رقم مه...
1       مع فيدرر يا آجا والكبار  وجه مبتسم بعيون قلب  
2    الداعون لمبدأ الاختلاط بين الجنسين كالداعين لإ...
3    ihe 94 ya78m amooo5 badiajnikhar Oukasafa reos...
4    قل شرق حلب ولا تقل حلب الشرقية وقل غرب حلب ولا...
5    طبيب المقاصة إصابة أحمد الشيخ بسيطة تحتاج للرا...
6    مرسي  مش هنام اكتر من اربع ساعات في اليوم لو ن...
7                                  انتخبوا البرص مشكلة
8    Retweeted هاشتاق العرب TheArabHashفيديو  أهالي...
9     مربع صغير أسود محافظ العاصمة تفقد سير العملية...
Name: tweet, dtype: str

### Translation

In [36]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,د محمود العلايليأرى أن الفريق أحمد شفيق رقم مه...,False,NEU,msa,0
1,مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب,False,NEU,msa,1
2,الداعون لمبدأ الاختلاط بين الجنسين كالداعين لإ...,True,NEG,msa,2
3,ihe 94 ya78m amooo5 badiajnikhar Oukasafa reos...,True,NEG,gulf,3
4,قل شرق حلب ولا تقل حلب الشرقية وقل غرب حلب ولا...,False,NEU,msa,4
5,طبيب المقاصة إصابة أحمد الشيخ بسيطة تحتاج للرا...,False,NEG,msa,5
6,مرسي مش هنام اكتر من اربع ساعات في اليوم لو ن...,True,NEG,egypt,6
7,انتخبوا البرص مشكلة,True,NEG,egypt,7
8,Retweeted هاشتاق العرب TheArabHashفيديو أهالي...,False,POS,msa,8
9,مربع صغير أسود محافظ العاصمة تفقد سير العملية...,False,NEU,msa,9


In [37]:
from googletrans import Translator
import re 

translator = Translator()

def translate_text(text):
    latin_words = re.findall(r'[a-zA-Z0-9]+', text)

    for la in latin_words:
        if la in text:
            try:
                translated_word = translator.translate(la, src='en', dest='ar').text
                text = re.sub(rf'\b{re.escape(la)}\b', translated_word, text)
            except Exception:
                continue
    return text

full['tweet'] = full['tweet'].apply(translate_text)

C:\Users\user\AppData\Local\Temp\ipykernel_9628\2925514532.py:15: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  continue


In [38]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,د محمود العلايليأرى أن الفريق أحمد شفيق رقم مه...,False,NEU,msa,0
1,مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب,False,NEU,msa,1
2,الداعون لمبدأ الاختلاط بين الجنسين كالداعين لإ...,True,NEG,msa,2
3,ihe 94 ya78m amooo5 badiajnikhar Oukasafa reos...,True,NEG,gulf,3
4,قل شرق حلب ولا تقل حلب الشرقية وقل غرب حلب ولا...,False,NEU,msa,4
5,طبيب المقاصة إصابة أحمد الشيخ بسيطة تحتاج للرا...,False,NEG,msa,5
6,مرسي مش هنام اكتر من اربع ساعات في اليوم لو ن...,True,NEG,egypt,6
7,انتخبوا البرص مشكلة,True,NEG,egypt,7
8,Retweeted هاشتاق العرب TheArabHashفيديو أهالي...,False,POS,msa,8
9,مربع صغير أسود محافظ العاصمة تفقد سير العملية...,False,NEU,msa,9


### Normalisation de text

In [39]:
def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "ء", text)
    text = re.sub("ئ", "ء", text)
    text = re.sub("ة", "ه", text)
    text = re.sub("گ", "ك", text)
    return text

In [40]:
full['tweet'] = full['tweet'].apply(normalize_arabic)

### remove latin untranslated words

In [41]:
def delete_latin(text):
    latin_words = re.findall(r'[a-zA-Z0-9]+', text)

    for la in latin_words:
        if la in text:
            text = re.sub(rf'\b{re.escape(la)}\b'," ", text)
    return text

full["tweet"] = full["tweet"].apply(delete_latin)

In [42]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,د محمود العلايلياري ان الفريق احمد شفيق رقم مه...,False,NEU,msa,0
1,مع فيدرر يا اجا والكبار وجه مبتسم بعيون قلب,False,NEU,msa,1
2,الداعون لمبدا الاختلاط بين الجنسين كالداعين لا...,True,NEG,msa,2
3,مساكين من الصبح و هوما رايحين ...,True,NEG,gulf,3
4,قل شرق حلب ولا تقل حلب الشرقيه وقل غرب حلب ولا...,False,NEU,msa,4
5,طبيب المقاصه اصابه احمد الشيخ بسيطه تحتاج للرا...,False,NEG,msa,5
6,مرسي مش هنام اكتر من اربع ساعات في اليوم لو ن...,True,NEG,egypt,6
7,انتخبوا البرص مشكله,True,NEG,egypt,7
8,هاشتاق العرب TheArabHashفيديو اهالي القدس ي...,False,POS,msa,8
9,مربع صغير اسود محافظ العاصمه تفقد سير العمليه...,False,NEU,msa,9


### Data split

In [43]:
train, val = train_test_split(full, test_size=0.10, random_state=42,
stratify=full["sarcasm"])

### Cleaned data export

In [48]:
from pathlib import Path

output_dir = Path(r"C:\ghoufrane\Master\M2\llms\Hanwritten-code-For-LLM-FineTuning\Data\cleaned_splits")
output_file_train = Path(f"{output_dir}\processed_train.csv")
output_file_val = Path(f"{output_dir}\processed_val.csv")

train 

train.drop(columns=["tweet"]).to_csv(output_file_train,index=False)
val.drop(columns=["tweet"]).to_csv(output_file_val, index=False)